# Compose Figure 4.2 and 4.3 (with basemap + vector fires)

**This composition provides:**
1. OpenStreetMap basemap under each panel (cities, roads visible)
2. Fire-reference polygons displayed as vectors (from shapefile, not rasterized)
3. Recurrence panel labeled clearly as derived from the fire-reference inventory (not model)

**Inputs (from `thesis_maps/` folder in Google Drive):**
- 7 GeoTIFFs: severity, persistence, recovery_gap, recurrence, ris_ecological, ris_fire, ris_balanced
- 2 shapefiles: fig_gt_fires (+ .dbf, .prj, .shx), fig_roi_polygon (same)

**Outputs (saved to same folder):**
- `figure_4_2_indicators.png` (300 DPI)
- `figure_4_3_ris_composites.png` (300 DPI)

In [ ]:
# --- Install dependencies (Colab) ---
!pip install -q rasterio geopandas matplotlib contextily


In [ ]:
# --- Mount Drive ---
from google.colab import drive
drive.mount('/content/drive')

DRIVE_FOLDER = '/content/drive/MyDrive/thesisv4/data/thesis_maps'
OUTPUT_FOLDER = DRIVE_FOLDER

import os
print(f'Files in {DRIVE_FOLDER}:')
for f in sorted(os.listdir(DRIVE_FOLDER)):
    print(f'  {f}')


In [ ]:
# --- Imports ---
import numpy as np
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

# Publication-quality defaults
plt.rcParams.update({
    'font.family':      'sans-serif',
    'font.sans-serif':  ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size':        11,
    'axes.titlesize':   12,
    'axes.labelsize':   10,
    'figure.dpi':       100,
    'savefig.dpi':      300,
    'savefig.bbox':     'tight',
    'savefig.facecolor':'white',
})


In [ ]:
# --- Load vector data (ROI + fire-reference polygons) ---
roi_gdf = gpd.read_file(os.path.join(DRIVE_FOLDER, 'fig_roi_polygon.shp'))
gt_fires_gdf = gpd.read_file(os.path.join(DRIVE_FOLDER, 'fig_gt_fires.shp'))

# Reproject to Web Mercator (EPSG:3857) for use with contextily basemaps
roi_wm = roi_gdf.to_crs('EPSG:3857')
gt_fires_wm = gt_fires_gdf.to_crs('EPSG:3857')

print(f'ROI features: {len(roi_gdf)}')
print(f'GT fires: {len(gt_fires_gdf)} polygons')
print(f'ROI bounds (WGS84): {roi_gdf.total_bounds}')
print(f'ROI bounds (Web Mercator): {roi_wm.total_bounds}')


In [ ]:
# --- Load rasters + reproject to Web Mercator for basemap compatibility ---
from rasterio.warp import reproject, Resampling, calculate_default_transform

def load_and_reproject(name, target_crs='EPSG:3857'):
    """Load a raster and reproject to target CRS (default: Web Mercator)."""
    path = os.path.join(DRIVE_FOLDER, f'fig_{name}.tif')
    with rasterio.open(path) as src:
        # Compute the target transform for the reprojection
        transform, width, height = calculate_default_transform(
            src.crs, target_crs, src.width, src.height, *src.bounds
        )
        # Allocate output array
        dst = np.full((height, width), np.nan, dtype=np.float32)

        # Reproject
        reproject(
            source        = rasterio.band(src, 1),
            destination   = dst,
            src_transform = src.transform,
            src_crs       = src.crs,
            dst_transform = transform,
            dst_crs       = target_crs,
            resampling    = Resampling.nearest,
            src_nodata    = src.nodata,
            dst_nodata    = np.nan,
        )

        # Compute the extent in target CRS for imshow
        left    = transform.c
        top     = transform.f
        right   = left + transform.a * width
        bottom  = top + transform.e * height
        extent  = [left, right, bottom, top]

    return dst, extent


# Load and reproject all rasters
raster_names = [
    'severity', 'persistence', 'recovery_gap', 'model_recurrence',
    'ris_ecological', 'ris_fire', 'ris_balanced',
]

indicators = {}
for name in raster_names:
    data, extent = load_and_reproject(name)
    indicators[name] = {'data': data, 'extent': extent}
    print(f'{name:20s} shape={data.shape}  '
          f'range=[{np.nanmin(data):.3f}, {np.nanmax(data):.3f}]  '
          f'valid={np.sum(~np.isnan(data)):,}')


In [ ]:
# --- Shared colormap ---
restoration_cmap = LinearSegmentedColormap.from_list(
    'restoration',
    ['#2c7bb6', '#abd9e9', '#ffffbf', '#fdae61', '#d7191c'],
    N=256,
)

# Fire polygon color
FIRE_ORANGE = '#ff6b00'

# ROI outline styling
ROI_EDGECOLOR = '#000000'
ROI_LINEWIDTH = 1.0


In [ ]:
# --- Plotting function with basemap ---
def plot_panel_with_basemap(ax, data, extent, title, label,
                              roi_gdf_wm, cmap=restoration_cmap,
                              overlay_gdf=None, overlay_color=None,
                              overlay_label=None):
    """Plot one panel:
       1. OSM basemap underneath
       2. ROI outline (thin black line)
       3. Data raster overlay (semi-transparent to show basemap through)
       4. Optional vector overlay (fire-reference polygons)

    ax must be in Web Mercator projection.
    """
    # 1. Set extent from ROI bounds (with small buffer)
    minx, miny, maxx, maxy = roi_gdf_wm.total_bounds
    buffer = (maxx - minx) * 0.02
    ax.set_xlim(minx - buffer, maxx + buffer)
    ax.set_ylim(miny - buffer, maxy + buffer)

    # 2. Add OSM basemap FIRST (bottom layer)
    try:
        cx.add_basemap(ax, crs='EPSG:3857',
                       source=cx.providers.CartoDB.Positron,
                       zoom=10, attribution_size=6)
    except Exception as e:
        print(f'Warning: basemap load failed for panel {label}: {e}')

    # 3. Data raster overlay (semi-transparent)
    if data is not None:
        im = ax.imshow(data, extent=extent, cmap=cmap,
                       vmin=0, vmax=1,
                       origin='upper', interpolation='nearest',
                       alpha=0.75, zorder=2)
    else:
        im = None

    # 4. Optional vector overlay (e.g., fire-reference polygons)
    if overlay_gdf is not None:
        overlay_gdf.plot(ax=ax, color=overlay_color,
                          edgecolor='#8b0000', linewidth=0.3,
                          alpha=0.85, zorder=3)

    # 5. ROI outline on top
    roi_gdf_wm.boundary.plot(ax=ax, color=ROI_EDGECOLOR,
                              linewidth=ROI_LINEWIDTH, zorder=4)

    # 6. Title and label
    ax.set_title(title, fontsize=11, pad=6)
    ax.text(0.02, 0.98, label,
            transform=ax.transAxes,
            fontsize=14, fontweight='bold',
            va='top', ha='left',
            bbox=dict(facecolor='white', edgecolor='black',
                      boxstyle='round,pad=0.3', alpha=0.9),
            zorder=5)

    # 7. Remove tick labels (coordinates in Web Mercator are large numbers)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect('equal')

    return im


## Figure 4.2 - Individual indicators (2x2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 13))

# Note: Recurrence panel is model-derived (from classification head), consistent with panels (a-c)
panels = [
    ('severity',     '(a) Severity (model)',                 axes[0, 0]),
    ('persistence',  '(b) Persistence (model)',              axes[0, 1]),
    ('recovery_gap', '(c) Recovery gap (model)',             axes[1, 0]),
    ('model_recurrence', '(d) Recurrence (model-derived)',    axes[1, 1]),
]

last_im = None
for name, title, ax in panels:
    label = title.split(' ')[0]
    title_only = ' '.join(title.split(' ')[1:])
    im = plot_panel_with_basemap(
        ax,
        indicators[name]['data'],
        indicators[name]['extent'],
        title_only,
        label,
        roi_wm,
    )
    last_im = im

# Shared colorbar at the bottom
fig.subplots_adjust(bottom=0.08, wspace=0.05, hspace=0.1)
cbar_ax = fig.add_axes([0.15, 0.03, 0.7, 0.02])
cbar = fig.colorbar(last_im, cax=cbar_ax, orientation='horizontal')
cbar.set_label('Restoration priority (0 = low, 1 = high)', fontsize=11)
cbar.ax.tick_params(labelsize=9)

out_path = os.path.join(OUTPUT_FOLDER, 'figure_4_2_indicators.png')
fig.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'\nSaved: {out_path}')


## Figure 4.3 - RIS composites + fire-reference overlay (2x2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 13))

# Panels (a)-(c): RIS composites (rasters)
# Panel (d): fire-reference polygons (vector, no raster, overlay only)

# (a) RIS Ecological
im1 = plot_panel_with_basemap(
    axes[0, 0],
    indicators['ris_ecological']['data'],
    indicators['ris_ecological']['extent'],
    'RIS Ecological priority', '(a)',
    roi_wm,
)

# (b) RIS Fire-focused
im2 = plot_panel_with_basemap(
    axes[0, 1],
    indicators['ris_fire']['data'],
    indicators['ris_fire']['extent'],
    'RIS Fire-focused', '(b)',
    roi_wm,
)

# (c) RIS Balanced
im3 = plot_panel_with_basemap(
    axes[1, 0],
    indicators['ris_balanced']['data'],
    indicators['ris_balanced']['extent'],
    'RIS Balanced', '(c)',
    roi_wm,
)

# (d) fire-reference polygons (VECTOR, no raster)
plot_panel_with_basemap(
    axes[1, 1],
    None,   # no raster
    None,   # no extent needed
    'Ground-truth fire polygons', '(d)',
    roi_wm,
    overlay_gdf=gt_fires_wm,
    overlay_color=FIRE_ORANGE,
)

# Shared colorbar for RIS panels
fig.subplots_adjust(bottom=0.08, wspace=0.05, hspace=0.1)
cbar_ax = fig.add_axes([0.15, 0.03, 0.5, 0.02])
cbar = fig.colorbar(im3, cax=cbar_ax, orientation='horizontal')
cbar.set_label('RIS (0 = low, 1 = high priority)', fontsize=11)
cbar.ax.tick_params(labelsize=9)

# Small legend for the fire-reference panel
gt_legend_ax = fig.add_axes([0.7, 0.02, 0.2, 0.04])
gt_legend_ax.axis('off')
n_fires = len(gt_fires_wm)
legend_elements = [Patch(facecolor=FIRE_ORANGE, edgecolor='#8b0000',
                          label=f'Fire polygons (n = {n_fires:,})')]
gt_legend_ax.legend(handles=legend_elements, loc='center', fontsize=9,
                     frameon=True, edgecolor='black')

out_path = os.path.join(OUTPUT_FOLDER, 'figure_4_3_ris_composites.png')
fig.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'\nSaved: {out_path}')


## Done!

Both figures are saved as 300 DPI PNGs in your Drive `thesis_maps/` folder:

- `figure_4_2_indicators.png` - 4 indicators over OSM basemap; recurrence panel clarifies that recurrence comes from the fire-reference inventory
- `figure_4_3_ris_composites.png` - 3 RIS profiles + accurate polygon overlay of the fire-reference polygons

Ready to insert into the thesis.